# Qwen3-VL Invoice Extraction — Standalone

Only the Qwen3-VL invoice extraction service. Qwen3-4B/accounting is in the separate notebook.

**API:** `POST /api/infer/extract-invoice`

## 1. Install dependencies

In [ ]:
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes pydantic python-dateutil "qwen-vl-utils[decord]" "pillow>=10.1" pymupdf fastapi uvicorn nest_asyncio pyngrok

## 2. Imports

In [ ]:
import base64
import copy
import io
import json
import os
import re
import tempfile
import time
import datetime
import asyncio
import threading
import traceback
from datetime import timedelta
from typing import Optional, List, Dict, Any, Union

import torch
from PIL import Image
from pydantic import BaseModel, Field, ValidationError, create_model
from dateutil import parser as dateparser
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

torch.manual_seed(0)

# Concurrency lock for safe GPU inference
inference_lock = asyncio.Lock()

# Unique Request ID generator (VLM-YYYYMMDD-HHMMSS-NNNN)
_vlm_request_counter = 0
_vlm_counter_lock = threading.Lock()

def generate_vlm_request_id() -> str:
    global _vlm_request_counter
    with _vlm_counter_lock:
        _vlm_request_counter += 1
        now_str = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        return f"VLM-{now_str}-{_vlm_request_counter:04d}"


## 3. Load Qwen3-VL-4B-Instruct (4-bit)

In [ ]:
MODEL_NAME_VL = "Qwen/Qwen3-VL-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1600 * 28 * 28

processor = AutoProcessor.from_pretrained(MODEL_NAME_VL, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
hf_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME_VL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16
)
hf_model.eval()

print("Loaded:", MODEL_NAME_VL)
print("4-bit quantization: enabled")


## 4. Invoice schema

In [ ]:
class BankDetails(BaseModel):
    account_holder_name: Optional[str] = None
    account_number: Optional[str] = None
    ifsc_code: Optional[str] = None
    bank_name: Optional[str] = None
    branch: Optional[str] = None
    upi_id: Optional[str] = None


class LineItem(BaseModel):
    description: Optional[str] = None
    hsn_code: Optional[str] = None          # official HSN/SAC tax code, always numeric
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    discount: Optional[float] = None
    taxable_amount: Optional[float] = None  # pre-tax line amount
    cgst_rate: Optional[float] = None       # percentage, e.g. 9 for 9%
    cgst_amount: Optional[float] = None
    sgst_rate: Optional[float] = None
    sgst_amount: Optional[float] = None
    igst_rate: Optional[float] = None
    igst_amount: Optional[float] = None
    total: Optional[float] = None           # post-tax line amount


class InvoiceSchema(BaseModel):
    invoice_number: Optional[str] = None
    invoice_date: Optional[str] = None
    due_date: Optional[str] = None
    po_number: Optional[str] = None

    vendor_name: Optional[str] = None
    vendor_address: Optional[str] = None
    vendor_gstin: Optional[str] = None
    vendor_pan: Optional[str] = None
    vendor_cin: Optional[str] = None
    vendor_phone: Optional[str] = None
    vendor_email: Optional[str] = None

    customer_name: Optional[str] = None
    customer_address: Optional[str] = None
    customer_gstin: Optional[str] = None
    customer_pan: Optional[str] = None

    payment_terms: Optional[str] = None
    bank_details: Optional[BankDetails] = None

    line_items: List[LineItem] = Field(default_factory=list)

    subtotal: Optional[float] = None
    discount_total: Optional[float] = None
    tax_total: Optional[float] = None
    shipping_charges: Optional[float] = None
    other_charges: Optional[float] = None
    round_off: Optional[float] = None
    total_amount: Optional[float] = None
    currency: Optional[str] = None

    additional_fields: Dict[str, Any] = Field(default_factory=dict)


# Header/line-items split kept from revision 1 -- not for constrained-decoding
# performance anymore (outlines is gone), but because a focused prompt+schema
# per call (header fields vs. the items table) still helps accuracy on dense
# tables. Derived with create_model so it can't drift out of sync with
# InvoiceSchema.
from pydantic import create_model

_header_field_defs = {
    name: (field.annotation, field)
    for name, field in InvoiceSchema.model_fields.items()
    if name != "line_items"
}
InvoiceHeaderSchema = create_model("InvoiceHeaderSchema", **_header_field_defs)

_header_schema_example = InvoiceHeaderSchema().model_dump()
HEADER_SCHEMA_JSON_EXAMPLE = json.dumps(_header_schema_example, indent=2)
LINE_ITEM_SCHEMA_JSON_EXAMPLE = json.dumps(LineItem().model_dump(), indent=2)

CORE_FIELDS = [f for f in InvoiceSchema.model_fields.keys() if f not in ("line_items", "additional_fields")]

# Schema example shown to the model includes ONE fully-populated line item (not an
# empty list) -- otherwise the model never sees the real field names and invents
# its own column names instead.
_schema_example = InvoiceSchema().model_dump()
_schema_example["line_items"] = [LineItem().model_dump()]
SCHEMA_JSON_EXAMPLE = json.dumps(_schema_example, indent=2)
print(SCHEMA_JSON_EXAMPLE)

## 5. Extraction prompts

In [ ]:
SYSTEM_PROMPT_HEADER = """You are a precise invoice-data-extraction engine reading an invoice IMAGE directly.

Rules:
1. Invoices vary widely in layout, field labels, language, and image quality. Do NOT assume any fixed position or label wording -- read the image semantically, using table structure and spatial layout to disambiguate which number belongs to which field.

2. "vendor" means the ISSUING/SELLING party -- the entity that generated and is sending this invoice, normally in the letterhead at the top. "customer" means whichever party is being BILLED / charged, i.e. whoever owes the money on this invoice. Do NOT rely on the literal label to decide this -- labels like "Bill To", "Sold By", "Party Name", or even "Ship To" are used inconsistently across templates. Reason about which party is issuing the invoice and which party is being charged, not which label is attached. vendor_gstin and customer_gstin should normally be DIFFERENT values.

3. The pre-tax total of all line items may be labeled "Subtotal", "Taxable Amount", "Taxable Value", or "Assessable Value" depending on the template -- these are the same concept. Map whichever one appears to the "subtotal" field.

4. If the invoice has an additional charge line that is NOT shipping/freight and NOT a line item (e.g. a service charge, handling fee, or similar), put its amount in "other_charges".

5. "bank_details" is a structured object, not free text: account_holder_name, account_number, ifsc_code, bank_name, branch, upi_id. Fill each sub-field independently from wherever it appears on the invoice (often a "Bank Details"/"Payment Details" box). Copy account_number, ifsc_code, and upi_id EXACTLY as printed. Leave a sub-field null if it isn't visible -- do not guess one from another.

6. "payment_terms" should be copied close to verbatim (e.g. "Net 30", "45 days from invoice date", "Due on receipt", "50% advance") -- do not paraphrase it, since the exact wording is used later to compute a due date when none is printed.

7. "due_date" is only for a due date actually PRINTED on the invoice. If none is printed, leave it null -- do not calculate one yourself.

8. customer_pan should be extracted the same way as vendor_pan if a customer PAN is visible on the invoice (independent of GSTIN).

9. Output ALL numbers as plain numbers with no currency symbols, no thousands separators (commas), and no units -- e.g. write 30134.00, not "Rs. 30,134.00".

10. If a schema field is not visible in the image, set it to null. NEVER invent, guess, or hallucinate a value.

11. If you see information that doesn't map to any schema field, put it under "additional_fields" as {"your_label": "value"} instead of discarding it.
11A. Extract EVERY meaningful piece of information visible in the invoice. If information does not map to a canonical field, preserve it under "additional_fields" using the original printed label whenever possible. NEVER discard information merely because the schema has no dedicated field.

11B. TAX EXTRACTION IS HIGH-PRIORITY. Identify and preserve every tax/accounting value and label visible anywhere on the page, including CGST, SGST, IGST, CESS, Input CGST, Input SGST, Input IGST, Input CESS, TDS, TCS, Tax Payable, GST Payable, Net Tax Payable, Tax Liability, tax rates, tax bases, tax amounts, deductions, credits, reversals, exemptions and other tax notes. "CGST" and "Input CGST" are different concepts. "TDS" is not GST.

11C. Put non-canonical tax/accounting information into "additional_fields.tax_details" while also preserving the original printed label/value. A tax detail should contain enough context to preserve the source meaning, for example:
{"label_as_printed":"Input CGST","semantic_type":"input_cgst","rate":null,"base_amount":null,"amount":1234.50,"source":"extracted"}
Do not invent a rate/base/amount that is not visible. Use null when not present.

11D. Preserve every other non-canonical field in additional_fields without overwriting unrelated keys.

12. Copy identifiers (GSTIN, PAN, CIN, phone, invoice number) EXACTLY as printed, character by character. Only dates get normalized, to YYYY-MM-DD.

13. This schema does NOT include line items -- they are extracted separately. Do not include a "line_items" key.

14. Output ONLY a single valid JSON object matching the schema below. No markdown fences, no commentary before or after.

Schema:
""" + HEADER_SCHEMA_JSON_EXAMPLE

SYSTEM_PROMPT_LINE_ITEMS = """You are a precise invoice-line-item-extraction engine reading an invoice IMAGE directly.

Rules:
1. Use the EXACT field names shown in the schema below. If a column shows an official HSN/SAC tax classification code (e.g. "997159", always numeric), put it in "hsn_code"; ignore any separate internal SKU/part code column if present -- it is not captured.

2. Extract EVERY line item row visible in the items table, top to bottom. "taxable_amount" is the line amount BEFORE tax; "total" is the final line amount AFTER tax.

2A. TAX COMPONENT EXTRACTION RULES (CRITICAL):
Identify and populate tax components ONLY from the ACTUAL PRINTED COLUMN HEADER / LABEL in the table:
- If the invoice explicitly has an "IGST" column / header (e.g. "IGST", "IGST %", "IGST Rate", "IGST Amount"): populate igst_rate and/or igst_amount.
- If the invoice explicitly has a "CGST" column / header (e.g. "CGST", "CGST %", "CGST Rate", "CGST Amount"): populate cgst_rate and/or cgst_amount.
- If the invoice explicitly has an "SGST" column / header (e.g. "SGST", "SGST %", "SGST Rate", "SGST Amount"): populate sgst_rate and/or sgst_amount.
- If the invoice has "CGST + SGST" or separate CGST and SGST columns: populate CGST and SGST fields accordingly.
- If the invoice table has only a generic header (such as "GST%", "GST Rate", "Tax%", "Tax Rate", "Tax Amount") without an explicit component label (CGST/SGST/IGST): DO NOT put that rate into cgst_rate, sgst_rate, or igst_rate, and DO NOT put that amount into cgst_amount, sgst_amount, or igst_amount. Leave all three components (cgst_rate/cgst_amount, sgst_rate/sgst_amount, igst_rate/igst_amount) as null. Put the generic tax rate/amount into that row's additional_fields if needed.
- NEVER assume GST% = CGST%.
- NEVER infer tax component from supplier state, customer state, place of supply, or GSTIN.
- Extract only what is visibly PRINTED on the invoice. Do NOT decide or guess the final tax treatment.
- If a tax component is not explicitly identifiable from printed labels: leave that component as null.
- NEVER populate cgst_rate/sgst_rate when the table only shows IGST. NEVER populate igst_rate when the table only shows CGST/SGST.

3. Output ALL numbers as plain numbers with no currency symbols, no thousands separators (commas), and no units.

4. If a schema field is not visible for a row, set it to null. NEVER invent, guess, or hallucinate a value.

5. Copy identifiers (HSN codes) EXACTLY as printed, character by character.
6. Preserve any row-level tax/accounting information that does not fit the canonical line-item schema in that row's "additional_fields", including printed labels such as Input CGST, TDS, TCS, CESS, tax payable and custom charges.

7. Output ONLY a single valid JSON ARRAY of line-item objects -- e.g. [{...}, {...}] -- one element per row, in the order they appear on the invoice. No markdown fences, no commentary before or after, no wrapping object.

Schema for each array element:
""" + LINE_ITEM_SCHEMA_JSON_EXAMPLE


def build_messages_header(image_path: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_HEADER},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
                {"type": "text", "text": "Extract the invoice header data from this image (everything except line items). Return only the JSON object."},
            ],
        },
    ]


def build_messages_line_items(image_path: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_LINE_ITEMS},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
                {"type": "text", "text": "Extract every line item row from this invoice's items table. Return only the JSON array."},
            ],
        },
    ]

## 6. VLM generation and JSON parsing

In [ ]:
def _extract_json_array_block(text):
    start = text.find("[")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "[":
            depth += 1
        elif text[i] == "]":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def _parse_line_items_freeform(raw_text):
    """Parses+repairs the free-form line-items array: strip fences, extract
    the bracketed block, parse, then normalize/validate each row -- dropping
    only the individual rows that don't validate rather than the whole list."""
    cleaned = strip_code_fences(raw_text)
    candidate = _extract_json_array_block(cleaned) or cleaned
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        return []
    if not isinstance(parsed, list):
        return []
    items = []
    for li in parsed:
        norm = normalize_line_item(li)
        try:
            items.append(LineItem(**norm).model_dump())
        except ValidationError:
            continue
    return items


def _run_generation(image_path: str, messages_fn, max_new_tokens: int) -> str:
    """One free-form generation call: builds the prompt from the given
    message-builder, runs the model, and returns the raw decoded text."""
    messages = messages_fn(image_path)
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(hf_model.device)

    # Some tokenizers don't define a pad token; fall back to eos so .generate()
    # doesn't raise or silently mis-pad.
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id

    with torch.no_grad():
        out = hf_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=pad_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return processor.batch_decode(out[:, input_len:], skip_special_tokens=True)[0]


def generate_invoice(image_path: str, max_new_tokens: int = 2048):
    """Two free-form generation calls -- header fields, then the line-items
    table -- each parsed and JSON-repaired independently.

    Returns (data_dict_or_None, raw_texts_dict). data is None only if the
    header call's output couldn't be parsed/repaired into anything usable --
    the caller falls back to rescue_via_reread in that case. raw_texts is
    always returned so a failure is still inspectable.
    """
    header_raw = _run_generation(image_path, build_messages_header, max_new_tokens)
    header_parsed = safe_json_parse(header_raw)
    header_data = loosely_validate_header(header_parsed) if header_parsed else None

    li_raw = _run_generation(image_path, build_messages_line_items, max_new_tokens)
    line_items = _parse_line_items_freeform(li_raw)

    raw_texts = {"header_raw": header_raw, "line_items_raw": li_raw}

    if header_data is None:
        return None, raw_texts

    header_data["line_items"] = line_items
    # Add structured lossless tax metadata without deleting any VLM fields.
    header_data = ensure_lossless_tax_metadata(header_data)
    return header_data, raw_texts

## 7. Normalization and repair helpers

In [ ]:
_FENCE_RE = re.compile(r"^```(?:json)?\s*|\s*```$", re.MULTILINE)

def strip_code_fences(text):
    return _FENCE_RE.sub("", text).strip()

def extract_json_block(text):
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None

def repair_truncated_json(text):
    """Best-effort repair for JSON truncated mid-generation:
    closes an open string, then closes any open brackets/braces."""
    s = text.strip()
    if not s.startswith("{"):
        start = s.find("{")
        if start == -1:
            return None
        s = s[start:]

    def scan(s):
        stack = []
        in_string = False
        escape = False
        for ch in s:
            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
                continue
            if ch == '"':
                in_string = True
            elif ch in "{[":
                stack.append(ch)
            elif ch in "}]":
                if stack:
                    stack.pop()
        return stack, in_string

    stack, in_string = scan(s)

    # If we ended mid-string, trim back to before the dangling key/value
    # and recompute the bracket stack on the trimmed text.
    if in_string:
        trim_at = max(s.rfind(",", 0, len(s)), s.rfind("{", 0, len(s)))
        s = s[:trim_at] if trim_at != -1 else s
        stack, in_string = scan(s)

    s = re.sub(r",\s*$", "", s)  # strip trailing comma before closing
    closers = {"{": "}", "[": "]"}
    s += "".join(closers[c] for c in reversed(stack))
    return s

def safe_json_parse(raw_text):
    cleaned = strip_code_fences(raw_text)
    for candidate in (cleaned, extract_json_block(cleaned)):
        if not candidate:
            continue
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue

    # NEW: last-resort attempt to repair truncated/malformed JSON
    repaired = repair_truncated_json(cleaned)
    if repaired:
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass

    return None

NUMERIC_TOP_FIELDS = {"subtotal", "discount_total", "tax_total", "shipping_charges",
                       "other_charges", "round_off", "total_amount"}
NUMERIC_LINE_ITEM_FIELDS = {"quantity", "unit_price", "discount",
                             "taxable_amount",
                             "cgst_rate", "cgst_amount",
                             "sgst_rate", "sgst_amount",
                             "igst_rate", "igst_amount",
                             "total"}

def _clean_numeric(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = v.strip()
        # Accounting convention: "(1,234.56)" means -1234.56.
        negative = s.startswith("(") and s.endswith(")")
        # FIX: Indian invoices commonly write amounts as "Rs.5000/-" where
        # "/-" means "only" (not a minus sign) -- strip that idiom first so
        # the trailing "-" isn't later mistaken for a negative sign.
        s = re.sub(r"/[-=]\s*$", "", s)
        # FIX: strip currency words/symbols (and any other letters) BEFORE the
        # digit/dot filter. Previously "Rs. 30,134.00" kept the period in
        # "Rs." (since "." passed the old filter untouched), producing the
        # unparseable string ".30134.00" and silently returning None for a
        # perfectly good number -- a real risk on Indian invoices where
        # "Rs." is a very common prefix.
        s = re.sub(r"[A-Za-z₹$€£]+\.?", "", s)
        cleaned = re.sub(r"[^\d.\-]", "", s)
        # FIX: collapse multiple decimal points -- keep only the last as the
        # true decimal separator (any earlier ones are noise, e.g. leftover
        # currency-abbreviation punctuation) instead of failing to parse.
        if cleaned.count(".") > 1:
            head, _, tail = cleaned.rpartition(".")
            cleaned = head.replace(".", "") + "." + tail
        if cleaned in ("", "-", ".", "-."):
            return None
        try:
            val = float(cleaned)
        except ValueError:
            return None
        return -abs(val) if negative else val
    return None

def _norm_key(k):
    return re.sub(r"[^a-z0-9]+", " ", str(k).lower()).strip()

# A bare "Discount" column header is far more commonly an absolute currency
# amount than a percentage on real invoices, so "discount"/"disc" map to the
# amount field `discount`, not a percentage field.
# Note "cgst"/"sgst"/"igst" bare aliases map to the *amount* fields --
# "cgst rate"/"cgst %"/"cgst pct" map separately to the *rate* fields below,
# so a table with both an amount column and a rate column keeps them apart.
LINE_ITEM_KEY_ALIASES = {
    "description": {"description", "product name", "product name description", "item",
                     "item description", "particulars", "product"},
    "hsn_code": {"hsn code", "hsn", "hsn sac", "sac"},
    "quantity": {"qty", "qty nos", "quantity", "nos", "pcs"},
    "unit_price": {"unit price", "rate", "price", "unit rate"},
    "discount": {"disc amt", "discount amt", "discount amount", "disc value", "discount"},
    "taxable_amount": {"taxable amt rs", "taxable amount", "taxable amt", "taxable value", "assessable value"},
    "cgst_rate": {"cgst rate", "cgst %", "cgst pct", "cgst percent"},
    "cgst_amount": {"cgst amount", "cgst amt", "cgst rs", "cgst"},
    "sgst_rate": {"sgst rate", "sgst %", "sgst pct", "sgst percent"},
    "sgst_amount": {"sgst amount", "sgst amt", "sgst rs", "sgst"},
    "igst_rate": {"igst rate", "igst %", "igst pct", "igst percent"},
    "igst_amount": {"igst amount", "igst amt", "igst rs", "igst"},
    "total": {"total rs", "total", "amount total", "line total", "amount", "item total"},
}
_ALIAS_LOOKUP = {alias: canon for canon, aliases in LINE_ITEM_KEY_ALIASES.items() for alias in aliases}

def normalize_line_item(item):
    if not isinstance(item, dict):
        return {}
    out = {}
    stray = {}
    for k, v in item.items():
        canon = _ALIAS_LOOKUP.get(_norm_key(k))
        if canon is None:
            if str(k) != "additional_fields":
                stray[str(k)] = v
            continue
        out[canon] = _clean_numeric(v) if canon in NUMERIC_LINE_ITEM_FIELDS else v

    # Never drop row-level information that has no canonical field.
    existing_extra = item.get("additional_fields")
    if isinstance(existing_extra, dict):
        stray = {**existing_extra, **stray}

    if stray:
        out["additional_fields"] = stray
    return out


def loosely_validate(data):
    """Repairs the free-form JSON dict returned by the model into a valid
    InvoiceSchema instance: cleans numeric fields, drops/quarantines fields
    that don't validate (into additional_fields) instead of failing the
    whole record."""
    if not isinstance(data, dict):
        return None

    working = dict(data)
    if isinstance(working.get("line_items"), list):
        working["line_items"] = [normalize_line_item(li) for li in working["line_items"] if isinstance(li, dict)]
    for f in NUMERIC_TOP_FIELDS:
        if f in working:
            working[f] = _clean_numeric(working[f])

    filtered = {k: v for k, v in working.items() if k in InvoiceSchema.model_fields}
    stray = {k: v for k, v in working.items() if k not in InvoiceSchema.model_fields}
    if stray:
        filtered.setdefault("additional_fields", {})
        if isinstance(filtered["additional_fields"], dict):
            filtered["additional_fields"].update(stray)

    for _ in range(len(InvoiceSchema.model_fields) + 2):
        try:
            return InvoiceSchema(**filtered).model_dump()
        except ValidationError as e:
            bad_fields = {err["loc"][0] for err in e.errors() if err.get("loc")}
            if not bad_fields:
                return None
            progressed = False
            for bf in bad_fields:
                if bf == "line_items":
                    kept = []
                    for li in filtered.get("line_items", []):
                        try:
                            kept.append(LineItem(**li).model_dump())
                        except ValidationError:
                            pass
                    if len(kept) != len(filtered.get("line_items", [])):
                        filtered["line_items"] = kept
                        progressed = True
                elif bf in filtered:
                    val = filtered.pop(bf)
                    filtered.setdefault("additional_fields", {})
                    if isinstance(filtered["additional_fields"], dict):
                        filtered["additional_fields"][f"unparsed_{bf}"] = val
                    progressed = True
            if not progressed:
                return None
    return None


def loosely_validate_header(data):
    """Same repair-and-drop strategy as loosely_validate, but against
    InvoiceHeaderSchema (no line_items) -- used for the header generation
    call now that both calls are free-form."""
    if not isinstance(data, dict):
        return None

    working = dict(data)
    for f in NUMERIC_TOP_FIELDS:
        if f in working:
            working[f] = _clean_numeric(working[f])

    filtered = {k: v for k, v in working.items() if k in InvoiceHeaderSchema.model_fields}
    stray = {k: v for k, v in working.items() if k not in InvoiceHeaderSchema.model_fields}
    if stray:
        filtered.setdefault("additional_fields", {})
        if isinstance(filtered["additional_fields"], dict):
            filtered["additional_fields"].update(stray)

    for _ in range(len(InvoiceHeaderSchema.model_fields) + 2):
        try:
            return InvoiceHeaderSchema(**filtered).model_dump()
        except ValidationError as e:
            bad_fields = {err["loc"][0] for err in e.errors() if err.get("loc")}
            if not bad_fields:
                return None
            progressed = False
            for bf in bad_fields:
                if bf in filtered:
                    val = filtered.pop(bf)
                    filtered.setdefault("additional_fields", {})
                    if isinstance(filtered["additional_fields"], dict):
                        filtered["additional_fields"][f"unparsed_{bf}"] = val
                    progressed = True
            if not progressed:
                return None
    return None

## 8. Identifier validation and VLM re-read fallback

In [ ]:
from datetime import timedelta

CODE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

PATTERNS = {
    "vendor_gstin": re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b"),
    "customer_gstin": re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b"),
    "vendor_pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "customer_pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "vendor_cin": re.compile(r"\b[UL]\d{5}[A-Z]{2}\d{4}[A-Z]{3}\d{6}\b"),
    "vendor_phone": re.compile(r"\b(?:\+?91[\-\s]?)?[6-9]\d{9}\b"),
    "vendor_email": re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
    "invoice_date": re.compile(r"\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b|\b\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2}\b"),
    "due_date": re.compile(r"\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b|\b\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2}\b"),
    # Bank sub-fields -- only used to fill a gap the model left null, never
    # to override an extracted value (see fill_missing_bank_fields).
    "ifsc_code": re.compile(r"\b[A-Z]{4}0[A-Z0-9]{6}\b"),
    "account_number": re.compile(r"\b\d{9,18}\b"),
    "upi_id": re.compile(r"\b[a-zA-Z0-9.\-_]{2,64}@[a-zA-Z]{2,64}\b"),
}

LABEL_HINTS = {
    "vendor_gstin": ["gstin", "gst no", "gst reg", "gst number", "gst in", "tax registration"],
    "customer_gstin": ["gstin", "gst no", "gst reg", "gst number"],
    "vendor_pan": ["pan no", "pan number", "pan:", "permanent account number"],
    "customer_pan": ["pan no", "pan number", "pan:", "permanent account number"],
    "vendor_cin": ["cin", "cin no", "corporate identity"],
    "vendor_phone": ["phone", "mobile", "contact", "tel", "ph."],
    "vendor_email": ["email", "e-mail", "mail id"],
    "invoice_date": ["invoice date", "date", "dated", "bill date", "document date", "issue date"],
    "due_date": ["due date", "payment due", "payable by", "pay by"],
    "ifsc_code": ["ifsc", "ifsc code", "ifs code"],
    "account_number": ["a/c no", "account no", "account number", "bank a/c", "acc no", "a c no"],
    "upi_id": ["upi", "upi id", "vpa"],
}

def is_valid_gstin_format(v):
    return bool(v) and bool(PATTERNS["vendor_gstin"].fullmatch(v.strip().upper()))

def is_valid_gstin_checksum(v):
    # Verified against two independently-published GSTIN Luhn mod-36 worked
    # examples (27AABCU9603R1ZN and 27AAPFU0939F1ZV) -- this formula is correct
    # as originally written. Left unchanged; do not "fix" this again without
    # re-verifying against a real worked example first.
    if not v or len(v) != 15:
        return False
    v = v.strip().upper()
    try:
        factor = 1
        total = 0
        for ch in v[:-1]:
            digit = CODE36.index(ch)
            code_point = factor * digit
            factor = 2 if factor == 1 else 1
            code_point = (code_point // 36) + (code_point % 36)
            total += code_point
        check_digit = CODE36[(36 - (total % 36)) % 36]
        return check_digit == v[-1]
    except (ValueError, IndexError):
        return False

def is_valid_pan_format(v):
    return bool(v) and bool(re.fullmatch(r"[A-Z]{5}\d{4}[A-Z]", v.strip().upper()))

def is_valid_ifsc_format(v):
    return bool(v) and bool(re.fullmatch(r"[A-Z]{4}0[A-Z0-9]{6}", v.strip().upper()))

# Only ifsc_code/account_number/upi_id have patterns reliable enough to
# regex-match on their own; bank_name/branch/account_holder_name are left
# exactly as the model returned them (see fill_missing_bank_fields).
BANK_FIELD_VALIDATORS = {
    "ifsc_code": is_valid_ifsc_format,
    "account_number": lambda v: bool(v) and v.strip().isdigit() and 9 <= len(v.strip()) <= 18,
    "upi_id": lambda v: bool(v) and bool(PATTERNS["upi_id"].fullmatch(v.strip())),
}

def derive_pan_from_gstin(gstin):
    if gstin and len(gstin) == 15:
        candidate = gstin[2:12].upper()
        if is_valid_pan_format(candidate):
            return candidate
    return None

def normalize_date(value):
    if not value:
        return None
    try:
        dt = dateparser.parse(value, dayfirst=True, fuzzy=True)
        return dt.strftime("%Y-%m-%d")
    except (ValueError, OverflowError, TypeError):
        return value

# "Net 30", "30 days from invoice date", "Due on receipt", "50% advance", etc.
# Deliberately simple -- anything it can't parse just leaves due_date null,
# same as before; it never guesses.
_NET_TERMS_RE = re.compile(r"net\s*(\d{1,3})|(\d{1,3})\s*days?", re.IGNORECASE)
_IMMEDIATE_TERMS = ("due on receipt", "immediate", "advance", "cash", "cod", "payable on receipt", "prepaid")

def compute_due_date_from_terms(payment_terms, invoice_date):
    """Best-effort due date from payment_terms text + invoice_date. Only ever
    used to FILL a missing due_date -- never overrides a due_date the model
    actually read off the invoice (see extract_invoice).

    NOTE: by the time this runs, invoice_date has already been through
    normalize_date() and is YYYY-MM-DD -- parse it WITHOUT dayfirst=True, or
    an unambiguous ISO date like 2026-07-01 gets misread as day=07 (dayfirst
    forces day-month-year even on year-first strings)."""
    if not payment_terms or not invoice_date:
        return None
    try:
        base = dateparser.parse(invoice_date)
    except (ValueError, OverflowError, TypeError):
        return None
    terms_lower = payment_terms.lower()
    if any(term in terms_lower for term in _IMMEDIATE_TERMS):
        return base.strftime("%Y-%m-%d")
    m = _NET_TERMS_RE.search(terms_lower)
    if m:
        days = int(m.group(1) or m.group(2))
        return (base + timedelta(days=days)).strftime("%Y-%m-%d")
    return None

def regex_fallback(field, raw_text, start_after=0):
    """Returns (matched_value_or_None, match_start_index_or_-1).

    Accepts `start_after` so a caller can force a second, related lookup
    (e.g. customer_gstin after vendor_gstin) to search past the first match
    instead of re-finding the same label/window -- otherwise vendor_gstin and
    customer_gstin can both resolve to the same text since they share label
    keywords like "gstin".
    """
    pattern = PATTERNS.get(field)
    if pattern is None or not raw_text:
        return None, -1
    lowered = raw_text.lower()
    for hint in LABEL_HINTS.get(field, []):
        idx = lowered.find(hint, start_after)
        if idx != -1:
            window = raw_text[max(0, idx - 10): idx + 120]
            m = pattern.search(window)
            if m:
                return m.group().strip(), idx
    m = pattern.search(raw_text, start_after)
    return (m.group().strip(), m.start()) if m else (None, -1)

TRANSCRIBE_PROMPT = (
    "Transcribe every piece of text visible in this image exactly as printed, "
    "line by line, including all labels, numbers, codes, and dates. Do not "
    "summarize, interpret, or omit anything. Plain text only -- no JSON, no "
    "commentary."
)

def _build_transcribe_messages(image_path):
    return [
        {"role": "system", "content": "You transcribe text from images verbatim."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
            {"type": "text", "text": TRANSCRIBE_PROMPT},
        ]},
    ]

def get_backup_text_via_reread(image_path):
    """Re-reads the invoice image with the loaded VLM, asking for a raw
    transcription. Used as the fallback text source for regex_fallback()
    wherever a field fails validation. Wrapped in try/except because this is
    already a fallback path -- a second failure here should degrade to an
    empty string, not crash the pipeline."""
    try:
        return _run_generation(image_path, _build_transcribe_messages, max_new_tokens=1024)
    except Exception:
        return ""

## 9. Invoice arithmetic checks

In [ ]:
def reconcile_line_item(item, tolerance=0.02):
    qty, price = item.get("quantity"), item.get("unit_price")
    stated_total = item.get("total")

    if qty is None or price is None or stated_total is None:
        return {"status": "incomplete"}

    gross = qty * price
    discount_amt = item.get("discount") or 0
    taxable_declared = item.get("taxable_amount")
    cgst = item.get("cgst_amount")
    sgst = item.get("sgst_amount")
    igst = item.get("igst_amount")

    taxable_computed = gross - discount_amt
    base = taxable_declared if taxable_declared is not None else taxable_computed

    # Per-tax-type rates (cgst_rate/sgst_rate/igst_rate), summed -- an extra
    # arithmetic-check candidate alongside the amount-based one below.
    rate_parts = [item.get("cgst_rate"), item.get("sgst_rate"), item.get("igst_rate")]
    split_rate = sum(r for r in rate_parts if r is not None) if any(r is not None for r in rate_parts) else None

    has_tax_info = any(v is not None for v in (cgst, sgst, igst)) or split_rate is not None

    if not has_tax_info:
        if abs(gross - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": "gross_equals_total_no_tax"}
        if taxable_declared is not None and abs(taxable_declared - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": "taxable_equals_total_no_tax"}
        if stated_total > base + tolerance:
            return {
                "status": "unverifiable_tax_breakdown",
                "pretax_amount": round(base, 2), "stated_total": stated_total,
                "implied_tax": round(stated_total - base, 2),
            }
        return {"status": "mismatch", "candidates": {"gross_only": round(gross, 2)}, "stated": stated_total}

    tax_total = sum(v for v in (cgst, sgst, igst) if v is not None)

    candidates = {"taxable_plus_tax": base + tax_total}
    if split_rate is not None:
        candidates["taxable_times_split_rates"] = base * (1 + split_rate / 100)

    for label, val in candidates.items():
        if abs(val - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": label, "computed": round(val, 2)}

    return {"status": "mismatch", "candidates": {k: round(v, 2) for k, v in candidates.items()}, "stated": stated_total}


def reconcile_invoice(data, tolerance=0.5):
    line_items = data.get("line_items") or []
    line_total_sum = sum(li.get("total") for li in line_items if isinstance(li.get("total"), (int, float)))

    # Only sum taxable_amount when EVERY line item has it -- falling back to
    # `total` (post-tax) per-item when taxable_amount is missing would mix
    # pre-tax and post-tax bases into one "taxable" sum, which isn't
    # apples-to-apples.
    taxable_vals = [li.get("taxable_amount") for li in line_items if isinstance(li.get("taxable_amount"), (int, float))]
    taxable_basis_complete = len(line_items) > 0 and len(taxable_vals) == len(line_items)
    line_taxable_sum = sum(taxable_vals) if taxable_vals else None

    subtotal = data.get("subtotal")
    discount_total = data.get("discount_total") or 0
    tax_total = data.get("tax_total") or 0
    shipping = data.get("shipping_charges") or 0
    other_charges = data.get("other_charges") or 0
    round_off = data.get("round_off") or 0
    total_amount = data.get("total_amount")

    result = {"line_items_sum": round(line_total_sum, 2) if line_items else None}

    if subtotal is not None and line_items:
        if taxable_basis_complete:
            result["subtotal_match"] = abs(line_taxable_sum - subtotal) <= tolerance
        else:
            result["subtotal_match"] = None
            result["subtotal_check_note"] = "incomplete_taxable_amount_on_line_items"

    if total_amount is not None:
        # subtotal is defined as the post-discount taxable value in the
        # prompt ("Taxable Amount"/"Taxable Value"/"Assessable Value"), so it
        # is already net of discount and discount_total is NOT subtracted a
        # second time here.
        base = subtotal if subtotal is not None else (line_taxable_sum if taxable_basis_complete else line_total_sum)
        reconstructed = base + tax_total + shipping + other_charges + round_off
        result["total_reconstructed"] = round(reconstructed, 2)
        result["total_match"] = abs(reconstructed - total_amount) <= tolerance
        if discount_total:
            result["total_reconciliation_note"] = (
                "discount_total not subtracted here -- subtotal is assumed already "
                "net of discount per the extraction prompt's field definitions"
            )

    return result

## 10. Lossless tax metadata helper

In [ ]:
def ensure_lossless_tax_metadata(invoice: dict) -> dict:
    """
    Add a structured tax_details layer while preserving the raw VLM
    additional_fields exactly. This is additive and lossless.
    """
    if not isinstance(invoice, dict):
        return invoice

    extras = invoice.get("additional_fields")
    if not isinstance(extras, dict):
        extras = {}
        invoice["additional_fields"] = extras

    tax_details = extras.get("tax_details")
    if not isinstance(tax_details, dict):
        tax_details = {}
        extras["tax_details"] = tax_details

    # Preserve any tax_details generated by the VLM. Also surface common
    # top-level/extra labels into structured semantic buckets without
    # deleting the originals.
    semantic_map = {
        "cgst": ("output_tax", "cgst"),
        "sgst": ("output_tax", "sgst"),
        "igst": ("output_tax", "igst"),
        "cess": ("output_tax", "cess"),
        "input cgst": ("input_tax_credit", "input_cgst"),
        "input sgst": ("input_tax_credit", "input_sgst"),
        "input igst": ("input_tax_credit", "input_igst"),
        "input cess": ("input_tax_credit", "input_cess"),
        "tds": ("tds", "amount"),
        "tcs": ("tcs", "amount"),
        "tax payable": ("tax_payable", "amount"),
        "gst payable": ("tax_payable", "amount"),
        "net tax payable": ("tax_payable", "amount"),
        "tax liability": ("tax_payable", "amount"),
    }

    def norm_label(label):
        return re.sub(r"[^a-z0-9]+", " ", str(label).lower()).strip()

    # Canonical line-level output tax information.
    output_tax = tax_details.setdefault("output_tax", {})
    for key, rate_key, amount_key in (
        ("cgst", "cgst_rate", "cgst_amount"),
        ("sgst", "sgst_rate", "sgst_amount"),
        ("igst", "igst_rate", "igst_amount"),
    ):
        rate = invoice.get(rate_key)
        amount = invoice.get(amount_key)
        if rate is not None or amount is not None:
            node = output_tax.setdefault(key, {})
            if rate is not None:
                node["rate"] = rate
            if amount is not None:
                node["amount"] = amount

    # Existing raw keys in additional_fields.
    for raw_key, raw_value in list(extras.items()):
        if raw_key == "tax_details":
            continue
        k = norm_label(raw_key)
        if k not in semantic_map:
            continue
        bucket, leaf = semantic_map[k]
        bucket_obj = tax_details.setdefault(bucket, {})
        if bucket == "tds":
            bucket_obj.setdefault("amount", raw_value)
        elif bucket == "tcs":
            bucket_obj.setdefault("amount", raw_value)
        elif bucket == "tax_payable":
            bucket_obj.setdefault("amount", raw_value)
        else:
            node = bucket_obj.setdefault(leaf, {})
            node.setdefault("amount", raw_value)

    # Always make these buckets available so downstream LLM logic sees a
    # stable semantic structure. Nulls mean "not observed", not zero.
    tax_details.setdefault("output_tax", {})
    tax_details.setdefault("input_tax_credit", {})
    tax_details.setdefault("tds", {})
    tax_details.setdefault("tcs", {})
    tax_details.setdefault("tax_payable", {})
    tax_details.setdefault("other_tax_information", [])

    return invoice

## 11. Extraction assembly

In [ ]:
STRUCTURED_VALIDATORS = {
    "vendor_gstin": lambda v: is_valid_gstin_format(v) and is_valid_gstin_checksum(v),
    "customer_gstin": lambda v: is_valid_gstin_format(v) and is_valid_gstin_checksum(v),
    "vendor_pan": is_valid_pan_format,
    "customer_pan": is_valid_pan_format,
    "vendor_cin": lambda v: bool(v) and bool(PATTERNS["vendor_cin"].fullmatch(v.strip().upper())),
    "vendor_phone": lambda v: bool(v) and bool(PATTERNS["vendor_phone"].fullmatch(v.strip())),
    "vendor_email": lambda v: bool(v) and bool(PATTERNS["vendor_email"].fullmatch(v.strip())),
}
REQUIRED_IDENTITY_FIELDS = ["vendor_name", "customer_name", "invoice_number", "total_amount"]


In [ ]:
def rescue_via_reread(image_path: str) -> dict:
    """Last-resort extraction when the model produced nothing usable at all
    (raised, or returned output that couldn't be parsed/validated). Re-reads
    the image with the VLM for a plain transcription and regex-matches every
    field we have a pattern for, so a total model failure still yields a
    partial, reviewable record instead of an empty one."""
    backup_text = get_backup_text_via_reread(image_path)
    data, field_sources = {}, {}
    gstin_match_pos = 0
    for field in PATTERNS:
        search_from = gstin_match_pos if field == "customer_gstin" else 0
        value, pos = regex_fallback(field, backup_text, start_after=search_from)
        if field == "vendor_gstin" and pos != -1:
            gstin_match_pos = pos + 1
        validator = STRUCTURED_VALIDATORS.get(field) or BANK_FIELD_VALIDATORS.get(field)
        if value and (validator is None or validator(value)):
            data[field] = value
            field_sources[field] = "reread_fallback"
    for date_field in ("invoice_date", "due_date"):
        if data.get(date_field):
            data[date_field] = normalize_date(data[date_field])
    return {"data": data, "field_sources": field_sources}

def fill_missing_bank_fields(data, get_backup_text_fn):
    """Bank sub-fields are only ever CORRECTED when the model failed to
    fetch that specific sub-field -- never overwritten if the model already
    returned a value. Only ifsc_code/account_number/upi_id have patterns
    reliable enough to regex-match; bank_name/branch/account_holder_name are
    left exactly as the model returned them."""
    bd = data.get("bank_details")
    if not isinstance(bd, dict):
        bd = {"raw_text": bd} if bd else {}

    missing = [f for f in ("ifsc_code", "account_number", "upi_id") if not bd.get(f)]
    if missing:
        backup_text = get_backup_text_fn()
        for field in missing:
            value, _ = regex_fallback(field, backup_text)
            validator = BANK_FIELD_VALIDATORS.get(field)
            if value and (validator is None or validator(value)):
                bd[field] = value

    data["bank_details"] = bd or None

def extract_invoice(image_path: str) -> dict:
    # Generation can legitimately fail (CUDA OOM, a bad frame from a
    # malformed PDF, a transient model error) -- surface a reviewable result
    # instead of crashing the run.
    try:
        data, raw_texts = generate_invoice(image_path)
    except Exception as e:
        rescue = rescue_via_reread(image_path)
        return {
            "data": rescue["data"] or None,
            "field_sources": rescue["field_sources"],
            "needs_review": True,
            "review_reasons": [f"generation_error:{type(e).__name__}: {e}", "reread_rescue_attempted"],
            "raw_output": None, "generation_path": "error_reread_rescue",
        }

    if data is None:
        rescue = rescue_via_reread(image_path)
        return {
            "data": rescue["data"] or None,
            "field_sources": rescue["field_sources"],
            "needs_review": True,
            "review_reasons": ["model_output_unparseable", "reread_rescue_attempted"],
            "raw_output": raw_texts, "generation_path": "free_form_header_plus_line_items_reread_rescue",
        }

    source_note = "free_form_header_plus_line_items"

    # Backup text (a plain re-transcription) is only ever fetched lazily, the
    # first time some field actually fails validation, and cached after that
    # so it runs at most once per invoice.
    _backup_text_cache = {}
    def get_backup_text():
        if "text" not in _backup_text_cache:
            _backup_text_cache["text"] = get_backup_text_via_reread(image_path)
        return _backup_text_cache["text"]

    review_reasons = []
    # Only mark a field "llm"-sourced if the model actually returned a
    # non-null value for it.
    field_sources = {f: "llm" for f in CORE_FIELDS if data.get(f) is not None}

    # Track where the vendor_gstin match landed on the page so the
    # customer_gstin fallback lookup (which shares label keywords like
    # "gstin") is forced to search past it instead of re-finding the same
    # text.
    gstin_match_pos = 0
    for field, validator in STRUCTURED_VALIDATORS.items():
        value = data.get(field)
        valid = bool(value) and validator(value)
        if not valid:
            search_from = gstin_match_pos if field == "customer_gstin" else 0
            fallback, pos = regex_fallback(field, get_backup_text(), start_after=search_from)
            if field == "vendor_gstin" and pos != -1:
                gstin_match_pos = pos + 1
            if fallback and validator(fallback):
                data[field] = fallback
                field_sources[field] = "reread_fallback"
            elif value:
                review_reasons.append(f"{field}_failed_validation")

    # Explicit cross-field invariant -- vendor and customer must never share
    # a GSTIN.
    if data.get("vendor_gstin") and data.get("vendor_gstin") == data.get("customer_gstin"):
        review_reasons.append("vendor_customer_gstin_identical")

    # PAN derived from GSTIN only when the model didn't already give us a
    # valid one -- vendor and customer get the same treatment.
    if not data.get("vendor_pan") or not is_valid_pan_format(data.get("vendor_pan")):
        derived_pan = derive_pan_from_gstin(data.get("vendor_gstin"))
        if derived_pan:
            data["vendor_pan"] = derived_pan
            field_sources["vendor_pan"] = "derived_from_gstin"

    if not data.get("customer_pan") or not is_valid_pan_format(data.get("customer_pan")):
        derived_customer_pan = derive_pan_from_gstin(data.get("customer_gstin"))
        if derived_customer_pan:
            data["customer_pan"] = derived_customer_pan
            field_sources["customer_pan"] = "derived_from_gstin"

    for date_field in ("invoice_date", "due_date"):
        if data.get(date_field):
            data[date_field] = normalize_date(data[date_field])

    # due_date is ONLY computed when the invoice genuinely doesn't show one --
    # never overrides a due_date the model actually read off the page.
    if not data.get("due_date"):
        computed_due = compute_due_date_from_terms(data.get("payment_terms"), data.get("invoice_date"))
        if computed_due:
            data["due_date"] = computed_due
            field_sources["due_date"] = "computed_from_payment_terms"

    # Bank sub-fields: only fills gaps, never overwrites what the model read.
    fill_missing_bank_fields(data, get_backup_text)

    line_item_checks = [reconcile_line_item(li) for li in data.get("line_items", [])]
    for i, chk in enumerate(line_item_checks):
        if chk["status"] == "mismatch":
            review_reasons.append(f"line_item_{i}_arithmetic_mismatch")
    unverifiable_count = sum(1 for c in line_item_checks if c["status"] == "unverifiable_tax_breakdown")
    if unverifiable_count:
        review_reasons.append(f"{unverifiable_count}_line_item(s)_missing_tax_breakdown")

    invoice_check = reconcile_invoice(data)
    if invoice_check.get("total_match") is False:
        review_reasons.append("invoice_total_mismatch")
    if invoice_check.get("subtotal_match") is False:
        review_reasons.append("invoice_subtotal_mismatch")

    # Basic completeness gate -- an invoice missing line items entirely, or
    # missing the fields an AP workflow can't function without, should
    # always be reviewable rather than silently passing.
    if not data.get("line_items"):
        review_reasons.append("no_line_items_extracted")
    for f in REQUIRED_IDENTITY_FIELDS:
        if not data.get(f):
            review_reasons.append(f"{f}_missing")

    return {
        "data": data,
        "field_sources": field_sources,
        "line_item_reconciliation": line_item_checks,
        "line_items_with_unverifiable_tax": unverifiable_count,
        "invoice_reconciliation": invoice_check,
        "needs_review": len(review_reasons) > 0,
        "review_reasons": review_reasons,
        "generation_path": source_note,
        "raw_output": raw_texts,
    }


## 12. FastAPI — extraction only

In [ ]:
from fastapi import FastAPI, HTTPException
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI(title="Qwen3-VL Invoice Extraction API")

class ExtractionRequest(BaseModel):
    image_base64: str

@app.get("/health")
async def health():
    return {"status": "ok", "service": "qwen3-vl-invoice-extraction"}

@app.post("/api/infer/extract-invoice")
async def process_invoice(req: ExtractionRequest):
    req_id = generate_vlm_request_id()
    req_start_t = time.time()
    now_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    current_stage = "REQUEST_RECEIPT"
    temp_path = None
    pdf_path = None

    # Section 3: Request Received
    print("\n" + "=" * 70)
    print("🟢 VLM REQUEST RECEIVED")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"Timestamp: {now_dt}")
    print("HTTP Method: POST")
    print("Endpoint: /api/infer/extract-invoice")
    print("Content Type: application/json")
    print("Status: RECEIVED")

    try:
        # Section 4: Request Validation
        print("\n🟡 STATUS: VALIDATING REQUEST")
        current_stage = "VALIDATING_REQUEST"
        if not req.image_base64 or not isinstance(req.image_base64, str):
            print("\n🔴 STATUS: REQUEST VALIDATION FAILED")
            print(f"Request ID: {req_id}")
            print("Error: Empty or invalid image_base64 payload")
            print("Stage: VALIDATING_REQUEST")
            raise HTTPException(status_code=400, detail="Empty image_base64 payload")

        raw_bytes = base64.b64decode(req.image_base64)
        if not raw_bytes:
            print("\n🔴 STATUS: REQUEST VALIDATION FAILED")
            print(f"Request ID: {req_id}")
            print("Error: Decoded base64 payload is empty")
            print("Stage: VALIDATING_REQUEST")
            raise HTTPException(status_code=400, detail="Empty image_base64 payload")

        print("🟢 STATUS: REQUEST VALIDATED")

        # Section 5: Input Preparation
        print("\n" + "=" * 70)
        print("🟡 VLM INPUT PREPARATION")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        current_stage = "INPUT_PREPARATION"

        file_size_kb = round(len(raw_bytes) / 1024, 2)
        dimensions_info = "N/A"
        page_count_info = "N/A"
        input_format_str = "Unknown"

        if raw_bytes[:4] == b"%PDF":
            input_format_str = "PDF Document"
            import fitz
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_pdf:
                tmp_pdf.write(raw_bytes)
                pdf_path = tmp_pdf.name
            doc = fitz.open(pdf_path)
            if len(doc) == 0:
                raise HTTPException(status_code=400, detail="PDF contains no pages")
            page_count_info = str(len(doc))
            page = doc.load_page(0)
            pix = page.get_pixmap(dpi=200, alpha=False)
            dimensions_info = f"{pix.width} x {pix.height} px"
            with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tmp_img:
                temp_path = tmp_img.name
            pix.save(temp_path)
            doc.close()
        else:
            input_format_str = "Image (PNG/JPEG)"
            image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
            dimensions_info = f"{image.width} x {image.height} px"
            page_count_info = "1"
            with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tmp_img:
                temp_path = tmp_img.name
            image.save(temp_path, format="PNG")

        print("\n---------------------- INPUT ----------------------")
        print("Filename: invoice_upload (base64 stream)")
        print(f"Content type: {'application/pdf' if raw_bytes[:4] == b'%PDF' else 'image/png'}")
        print(f"File size: {file_size_kb} KB ({len(raw_bytes)} bytes)")
        print(f"Input format: {input_format_str}")

        print(f"\nInput type: {input_format_str}")
        print("Image/file successfully loaded: Yes")
        print(f"Image dimensions if available: {dimensions_info}")
        print(f"Number of pages if applicable: {page_count_info}")
        print("\n🟢 STATUS: VLM INPUT READY")

        # Concurrency Queue status
        if inference_lock.locked():
            print("\n🟠 STATUS: WAITING FOR GPU / INFERENCE LOCK")
        else:
            print("\n🟡 STATUS: WAITING FOR QWEN3-VL")

        async with inference_lock:
            print("🔵 STATUS: GPU AVAILABLE — STARTING INFERENCE")
            current_stage = "INFERENCE_EXECUTION"

            # Section 6: Model Status (Started)
            infer_start_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            infer_start_t = time.time()
            print("\n" + "=" * 70)
            print("🔵 QWEN3-VL INFERENCE STARTED")
            print("=" * 70)
            print(f"Request ID: {req_id}")
            print(f"Model: {MODEL_NAME_VL}")
            print(f"Start time: {infer_start_dt}")
            print("🔵 STATUS: QWEN3-VL INFERENCE RUNNING")

            result = extract_invoice(temp_path)

            infer_end_t = time.time()
            infer_latency = round(infer_end_t - infer_start_t, 2)
            infer_end_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Section 6: Model Status (Completed)
            print("\n" + "=" * 70)
            print("🟢 QWEN3-VL INFERENCE COMPLETED")
            print("=" * 70)
            print(f"Request ID: {req_id}")
            print(f"End time: {infer_end_dt}")
            print(f"Inference latency: {infer_latency}s")

        # Section 7: Raw Model Output
        raw_output_data = result.get("raw_output")
        print("\n" + "=" * 70)
        print("📥 QWEN3-VL RAW OUTPUT")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        if isinstance(raw_output_data, dict):
            for k, v in raw_output_data.items():
                print(f"\n--- {k} ---")
                print(v)
        else:
            print(raw_output_data if raw_output_data else "Raw output processed into structured schema.")

        # Section 8: Parsing / Extraction
        print("\n🟡 STATUS: PARSING VLM OUTPUT")
        print("🟡 STATUS: NORMALIZING INVOICE JSON")
        extracted_data = result.get("data") or {}
        print("\n" + "=" * 70)
        print("📄 EXTRACTED INVOICE JSON")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(json.dumps(extracted_data, ensure_ascii=False, indent=2))

        # Section 9: Response Validation
        print("\n🟡 STATUS: VALIDATING RESPONSE")
        current_stage = "RESPONSE_VALIDATION"
        if not isinstance(result, dict) or "data" not in result:
            raise ValueError("Extraction output missing required 'data' field")
        print("🟢 STATUS: RESPONSE VALIDATED")

        # Section 10: Response Sent
        current_stage = "RESPONSE_PREPARATION"
        total_latency = round(time.time() - req_start_t, 2)
        print("\n" + "=" * 70)
        print("📤 RESPONSE SENT TO BACKEND")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print("\nStatus: SUCCESS")
        print(f"Inference latency: {infer_latency}s")
        print(f"Total request latency: {total_latency} seconds")
        print("🟢 STATUS: RESPONSE SENT TO BACKEND")

        # Section 11: Final Request Summary
        print("\n" + "=" * 70)
        print("✅ VLM REQUEST COMPLETED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print("Status: SUCCESS\n")
        print("Stages:")
        print("  ✅ Request received")
        print("  ✅ Request validated")
        print("  ✅ Input prepared")
        print("  ✅ Qwen3-VL inference completed")
        print("  ✅ Output parsed")
        print("  ✅ Invoice JSON generated")
        print("  ✅ Response validated")
        print("  ✅ Response sent\n")
        print(f"Inference latency: {infer_latency}s")
        print(f"Total latency: {total_latency}s")
        print("=" * 70 + "\n")

        return result

    except HTTPException:
        raise
    except Exception as exc:
        total_latency = round(time.time() - req_start_t, 2)
        print("\n" + "=" * 70)
        print("🔴 VLM REQUEST FAILED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print("Status: FAILED")
        print(f"Stage: {current_stage}")
        print(f"Error Type: {type(exc).__name__}")
        print(f"Error Message: {str(exc)}")
        print(f"Total Latency: {total_latency}s")
        print("\nTraceback:")
        traceback.print_exc()
        print("=" * 70 + "\n")
        raise HTTPException(status_code=500, detail=str(exc))
    finally:
        for p in (temp_path, pdf_path):
            if p and os.path.exists(p):
                try: os.remove(p)
                except OSError: pass


## 13. Start API through ngrok

In [ ]:
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
public_url = "http://0.0.0.0:8000"
ngrok_status = "NOT_CONFIGURED"

if NGROK_AUTH_TOKEN:
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    ngrok_status = "CONNECTED"

# Section 1 & 16: Server Startup Status & NGROK
print("\n" + "=" * 70)
print("🚀 VLM INFERENCE SERVER")
print("=" * 70)
print("Service: Qwen3-VL Invoice Extraction")
print("Status: RUNNING")
print("Local URL: http://0.0.0.0:8000")
print(f"Health: {public_url}/health")
print(f"Endpoint: {public_url}/api/infer/extract-invoice")
print(f"ngrok: {ngrok_status}")
print("=" * 70)

if ngrok_status == "CONNECTED":
    print("\n" + "=" * 70)
    print("🌐 NGROK")
    print("=" * 70)
    print("Status: CONNECTED")
    print(f"Public URL: {public_url}")
    print(f"Health URL: {public_url}/health")
    print(f"Inference endpoint: {public_url}/api/infer/extract-invoice")
    print("=" * 70)
    print("\n🟢 VLM SERVICE READY FOR BACKEND REQUESTS")

print("\n🟢 READY — WAITING FOR BACKEND REQUESTS\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()
